# Memory

From the [Memory documentation](../../../docs/source/store-and-retrieve/memory.md):

> *"The NeMo Agent toolkit Memory subsystem is designed to store and retrieve a user's conversation history, preferences, and other 'long-term memory.' This is especially useful for building stateful LLM-based applications that recall user-specific data or interactions across multiple steps."*

## Supported Memory Providers

NeMo Agent Toolkit includes the following memory providers (all available as plugins):

| Provider | Plugin | Description |
|----------|--------|-------------|
| **Mem0** | `nvidia-nat-mem0ai` | AI-powered memory with semantic search |
| **Redis** | `nvidia-nat-redis` | Fast key-value memory storage |
| **Zep** | `nvidia-nat-zep-cloud` | Long-term memory for AI assistants |

## What You'll Learn

1. Why memory matters
2. Mem0 memory configuration
3. Redis memory configuration  
4. Memory functions for agents

For more details, see the [Memory documentation](../../../docs/source/store-and-retrieve/memory.md).


In [1]:
import getpass
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

# Load environment variables from .env file
load_dotenv()

# Check for NVIDIA API key (required for Redis memory embedder)
nvidia_api_key = os.environ.get("NVIDIA_API_KEY")

if nvidia_api_key:
    print("✅ NVIDIA_API_KEY loaded")
else:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA_API_KEY (get one at https://build.nvidia.com/): ")
    if nvidia_api_key:
        os.environ["NVIDIA_API_KEY"] = nvidia_api_key
        print("✅ NVIDIA_API_KEY set")
    else:
        print("⚠️ NVIDIA_API_KEY not set - Redis memory will not work")


✅ NVIDIA_API_KEY loaded


## 1. Mem0 Memory

Mem0 provides AI-powered memory with semantic search capabilities:


In [2]:
# Check for Mem0 API key
mem0_api_key = os.environ.get("MEM0_API_KEY")

if not mem0_api_key:
    mem0_api_key = getpass.getpass("Enter your MEM0_API_KEY (or press Enter to skip): ")
    if mem0_api_key:
        os.environ["MEM0_API_KEY"] = mem0_api_key
        print("✅ MEM0_API_KEY set")
else:
    print("✅ MEM0_API_KEY loaded")

try:
    from nat.plugins.mem0ai.sdk import Mem0Memory

    if mem0_api_key:
        # Mem0 configuration
        mem0_memory = Mem0Memory(
            name="mem0",
        )
        print("✅ Mem0 Memory configured")
    else:
        mem0_memory = None
        print("⏭️ Skipping Mem0 (no API key)")
except ImportError:
    mem0_memory = None
    print("⚠️ Install with: uv pip install -e packages/nvidia_nat_mem0")


✅ MEM0_API_KEY loaded
✅ Mem0 Memory configured


## 2. Redis Memory

Redis provides fast, self-hosted memory storage with semantic search via embeddings.

Redis memory requires:
1. A Redis server running (e.g., `docker run -d -p 6379:6379 redis`)
2. An embedder for semantic search


In [3]:
from nat.embedder.sdk import NIMEmbedder

# Create embedder for Redis memory (uses NVIDIA NIM)
redis_embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",
    truncate="END",
    name="redis_embedder",
)
print(f"✅ Embedder created: {redis_embedder.model_name}")

try:
    from nat.plugins.redis.sdk import RedisMemory

    # Redis configuration with embedder
    redis_memory = RedisMemory(
        host="localhost",
        port=6379,
        embedder_obj=redis_embedder,  # Required: embedder for semantic search
        name="redis_memory",
    )
    print("✅ Redis Memory configured")
except ImportError:
    redis_memory = None
    print("⚠️ Install with: uv pip install -e packages/nvidia_nat_redis")


✅ Embedder created: nvidia/nv-embedqa-e5-v5
✅ Redis Memory configured


## 3. Memory Tools

Memory tools allow agents to add and retrieve memories:


In [4]:
from nat.tool.sdk import AddMemoryTool
from nat.tool.sdk import GetMemoryTool

# Use Mem0 for this example (or Redis if available)
memory_backend = mem0_memory if mem0_memory else redis_memory

if memory_backend:
    # Create memory tools
    add_memory = AddMemoryTool(
        nat_memory=memory_backend,
        name="add_memory",
        description="Store information about the user such as preferences, facts, or important details",
    )

    get_memory = GetMemoryTool(
        nat_memory=memory_backend,
        name="get_memory",
        description="Retrieve previously stored information about the user",
    )

    print(f"✅ Memory tools created using: {memory_backend.computed_name}")
else:
    add_memory = None
    get_memory = None
    print("⚠️ No memory backend available - skipping memory tools")


✅ Memory tools created using: mem0


## 4. Complete Example: Agent with Memory

Let's create an agent that can remember user information:


In [5]:
from nat.agent.sdk import NatReActAgent
from nat.llm.sdk import NimLLM
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

if add_memory and get_memory:
    # Create LLM
    llm = NimLLM(
        model_name="meta/llama-3.3-70b-instruct",
        temperature=0.0,
        name="nim_llm",
    )

    # Create tools including memory
    time_tool = CurrentTimeTool(name="current_time")
    tools = [time_tool, add_memory, get_memory]

    # Create agent with memory capabilities
    memory_agent = NatReActAgent(
        tools=tools,
        llm=llm,
        verbose=True,
        additional_instructions=(
            "You have access to memory tools. Use add_memory to store important "
            "information about the user. Use get_memory to recall previously stored information."
        ),
    )

    workflow = NatWorkflow(entrypoint=memory_agent)
    print(f"✅ Memory Agent created with {len(tools)} tools")
else:
    workflow = None
    print("⏭️ Skipping agent creation (no memory backend)")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Memory Agent created with 3 tools


### Test 1: Store a memory


In [6]:
# Store information about the user
if workflow:
    result = await workflow.prompt("Remember that my name is Alice and I love coffee.")
    print(f"🤖 Response:\n{result}")


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


🤖 Response:
Hello Alice, I've taken note that you love coffee. Is there anything else you'd like to chat about or share?


### Test 2: Recall the memory


In [7]:
# Ask the agent to recall stored information
if workflow:
    result = await workflow.prompt("What do you know about me?")
    print(f"🤖 Response:\n{result}")


🤖 Response:
I don't know anything about you yet. Our conversation just started. What would you like to talk about? I can store information about you in my memory if you'd like.


### Save Configuration


In [8]:
# Save configuration
if workflow:
    config_dir = Path("./configs")
    config_dir.mkdir(parents=True, exist_ok=True)

    config_path = config_dir / "memory_agent.yaml"
    workflow.save_to_config_file(config_path)
    print(f"📄 Saved to: {config_path}")


📄 Saved to: configs/memory_agent.yaml


## CLI Commands

```bash
# Run the memory agent
nat run --config_file configs/memory_agent.yaml --input "Remember my favorite color is blue"

# Ask about stored memories
nat run --config_file configs/memory_agent.yaml --input "What do you know about me?"
```

## Summary

✅ **Mem0** - AI-powered semantic memory (cloud-based)  
✅ **Redis** - Fast self-hosted memory with embeddings  
✅ **Memory tools** - `AddMemoryTool` and `GetMemoryTool`  
✅ **Agent integration** - Memory-enabled agents  

## Next Steps

- **[07_retrievers.ipynb](./07_retrievers.ipynb)** - RAG with retrievers
- **[09_configuration_guide.ipynb](./09_configuration_guide.ipynb)** - YAML configuration
